# Section 2 - Cleaning & authority joins

**Goal of this notebook.** Turn the raw contracts table into a clean modelling
base. Three decisions, each documented with evidence from the data:

1. **Missing `bids_received` (9.9%)** - drop for modelling, but first check
   whether the missingness is random or concentrated (it matters for what we can
   claim).
2. **Zero-bid rows (1.5%)** - keep or drop, with a stated reason.
3. **Join authority attributes** - add `type_group` and `region` only. The
   aggregate columns in the authorities file (`spent_eur`, `contracts`,
   `suppliers`, `avg_eur`) are **deliberately excluded** as leakage risks.

Findings and the decisions they imply are collected in the closing cell.

In [1]:
import pandas as pd

# Notebook lives in notebooks/, data is one level up in data/
contracts = pd.read_csv("../data/raw/sigma-contracts.csv")
authorities = pd.read_csv("../data/raw/sigma-authorities.csv")

## 1. Missing `bids_received`

In [2]:
n_total = len(contracts)
n_missing = contracts["bids_received"].isna().sum()
print(f"total rows:        {n_total:,}")
print(f"missing bids:      {n_missing:,}  ({n_missing / n_total:.1%})")

total rows:        193,161
missing bids:      19,042  (9.9%)


In [3]:
# Is the missingness random, or concentrated? Break it down by procedure.
contracts["_missing_bids"] = contracts["bids_received"].isna()

missing_by_procedure = (
    contracts.groupby("procedure")["_missing_bids"]
    .agg(missing_rate="mean", n_rows="count")
    .sort_values("missing_rate", ascending=False)
)
missing_by_procedure

,missing_rate,n_rows
procedure,,
Пряко / без обявление,0.135424,24235
Открита,0.125544,74221
Неизвестна,0.118578,12768
Състезание,0.110739,42677
Договаряне с покана,0.041376,4882
Друго,0.000000,6
Събиране на оферти,0.000000,34372


The missing-bid rate is **not** uniform. It ranges from ~0% for
some procedure types to ~13.5% for others - a spread far too wide to be random
noise around the 9.9% overall rate. So this is closer to *Missing Not At Random*
(MNAR): whether a bid count was recorded depends on the procedure type, which is
itself one of our features.

Drop the missing rows (no target is possible) but record in the
Limitations section that the analysis is conditional on a recorded bid count
and that recording correlates with procedure. We are not claiming the dropped
rows behave identically to the kept ones.

In [4]:
# Quick cross-check: does missingness also track funding source and contract kind?
print("By eu_funded:")
print(contracts.groupby("eu_funded")["_missing_bids"].mean().round(4).to_string())
print("\nBy kind:")
print(contracts.groupby("kind")["_missing_bids"].mean().round(4).to_string())

By eu_funded:
eu_funded
0    0.1026
1    0.0729

By kind:
kind
company       0.0973
consortium    0.1360


In [5]:
# Apply the drop.
before = len(contracts)
modelling = contracts.dropna(subset=["bids_received"]).copy()
modelling["bids_received"] = modelling["bids_received"].astype(int)
after = len(modelling)
print(f"dropped {before - after:,} rows with missing bids; {after:,} remain")

dropped 19,042 rows with missing bids; 174,119 remain


## 2. Zero-bid rows

In [6]:
# Confirm where the zero-bid rows land and quantify the effect on the target.
n_zero = (modelling["bids_received"] == 0).sum()
print(f"zero-bid rows in modelling set: {n_zero:,}  ({n_zero / len(modelling):.1%})")

modelling["single_bid"] = (modelling["bids_received"] == 1)

pos_rate_with = modelling["single_bid"].mean()
pos_rate_without = modelling.loc[modelling["bids_received"] != 0, "single_bid"].mean()
print(f"positive (single-bid) rate, keeping zeros:   {pos_rate_with:.4f}")
print(f"positive rate if zeros were dropped instead: {pos_rate_without:.4f}")
print(f"difference: {abs(pos_rate_with - pos_rate_without):.4f}")

zero-bid rows in modelling set: 2,845  (1.6%)
positive (single-bid) rate, keeping zeros:   0.4489
positive rate if zeros were dropped instead: 0.4563
difference: 0.0075


**Decision: keep them.** Reasoning:

- The target is binary "exactly one bid vs. not." A zero-bid contract genuinely
  is "not single-bid," so it belongs in the negative class by definition.
- Dropping 1.6% of rows to tidy an edge case would be discarding real signal for
  cosmetic reasons. A zero-bid award (e.g. a direct award with no formal bid
  recorded) is a legitimate low-competition outcome - arguably the most extreme
  end of the spectrum we care about.
- It changes the positive rate only marginally, and we verify that below rather
  than asserting it.

The alternative - treating 0 and 1 together as "low competition" - would be a
*different target*, and a defensible one but it is not the target this project
defined. We keep the definition fixed and note the choice.

## 3. Join authority attributes

In [7]:
# Pre-merge safety checks.
dup_keys = authorities["eik"].duplicated().sum()
print(f"duplicate eik in authorities (would multiply rows): {dup_keys}")

contract_auths = set(modelling["authority_eik"].unique())
known_auths = set(authorities["eik"].unique())
unmatched = contract_auths - known_auths
print(f"distinct authorities in contracts:        {len(contract_auths):,}")
print(f"of those, absent from authorities file:   {len(unmatched)}")

duplicate eik in authorities (would multiply rows): 0
distinct authorities in contracts:        4,391
of those, absent from authorities file:   0


We enrich each contract with two attributes of the contracting authority:

- `type_group` - what kind of body it is (municipality, hospital, ministry, …)
- `region` - its administrative region

**What we deliberately do NOT join:** the aggregate columns `spent_eur`,
`contracts`, `suppliers`, `avg_eur`. These summarise an authority's *entire
history* of awards. Several of them are computed from outcomes that overlap with
what we are trying to predict (e.g. supplier counts and average competition),
so feeding them in would leak target information backwards into the features.
We keep only the two descriptive, non-outcome attributes.

In [8]:
# Safe to merge: unique key, full coverage. Bring in only the two columns we trust.
keep_cols = ["eik", "type_group", "region"]
rows_before = len(modelling)

modelling = modelling.merge(
    authorities[keep_cols],
    left_on="authority_eik",
    right_on="eik",
    how="left",
).drop(columns="eik")

print(f"rows before merge: {rows_before:,}")
print(f"rows after merge:  {len(modelling):,}  (must be unchanged)")
assert len(modelling) == rows_before, "merge changed the row count!"
print("row count preserved ✓")

rows before merge: 174,119
rows after merge:  174,119  (must be unchanged)
row count preserved ✓


In [9]:
# Post-merge null check on the two joined columns.
print("nulls after join:")
print(f"  type_group: {modelling['type_group'].isna().sum():,}")
print(f"  region:     {modelling['region'].isna().sum():,}  "
      f"({modelling['region'].isna().mean():.1%})")

nulls after join:
  type_group: 0
  region:     10,908  (6.3%)


`type_group` joins cleanly with no nulls. `region` is missing for a few
percent of rows - these are authorities with no single regional seat
(national-level bodies: ministries, state agencies and companies). Rather than
drop them, we fill with an explicit `"Unknown"` category so the information
"region not applicable / not recorded" is preserved and usable by the model.

In [10]:
# Fill missing region with an explicit category instead of dropping rows.
modelling["region"] = modelling["region"].fillna("Unknown")
print("region nulls after fill:", modelling["region"].isna().sum())
print("\nregion value counts (top 10):")
print(modelling["region"].value_counts().head(10).to_string())

region nulls after fill: 0

region value counts (top 10):
region
София (столица)    53531
Пловдив            12850
Unknown            10908
Варна               8872
Стара Загора        8128
Бургас              7727
Благоевград         5676
Велико Търново      4641
Враца               4622
Плевен              4382


## 4. Save the cleaned modelling table

We drop the temporary helper column and write the result to
`data/processed/`. EDA, baselines and models (sections 4–8) read from this file,
not the raw CSV - so the cleaning decisions above are applied exactly once and
consistently everywhere downstream.

In [11]:
modelling = modelling.drop(columns="_missing_bids")

out_path = "../data/processed/contracts-modelling.csv"
modelling.to_csv(out_path, index=False)

print(f"saved {modelling.shape[0]:,} rows x {modelling.shape[1]} cols -> {out_path}")
print("\ncolumns:", list(modelling.columns))
modelling.head()

saved 174,119 rows x 17 cols -> ../data/processed/contracts-modelling.csv

columns: ['id', 'unp', 'subject', 'authority', 'authority_eik', 'contractor', 'contractor_eik', 'kind', 'sector_code', 'procedure', 'signed_at', 'value_eur', 'eu_funded', 'bids_received', 'single_bid', 'type_group', 'region']


,id,unp,subject,authority,authority_eik,contractor,contractor_eik,kind,sector_code,procedure,signed_at,value_eur,eu_funded,bids_received,single_bid,type_group,region
0,e:00001-2021-0001:106742:_:eik:201472746:1,00001-2021-0001,Доставка и монтаж на каскадна инсталация от 6 ...,ИКОНОМИЧЕСКИ УНИВЕРСИТЕТ - ВАРНА,000083619,МЕГА ЕЛЕКТРОНИКС-АП ООД,201472746.0,company,42.0,Събиране на оферти,2021-07-16,32138.785068,0,3,False,образование,Варна
1,e:00001-2021-0003:9643:_:eik:131468980:1,00001-2021-0003,Предоставяне на далекосъобщителна услуга чрез ...,ИКОНОМИЧЕСКИ УНИВЕРСИТЕТ - ВАРНА,000083619,"""А1 БЪЛГАРИЯ"" ЕАД",131468980.0,company,64.0,Събиране на оферти,2021-08-26,35789.920392,0,1,True,образование,Варна
2,e:00001-2021-0006:15251:_:eik:103765686:1,00001-2021-0006,"Приготвяне, доставка на храна и осъществяване ...",ИКОНОМИЧЕСКИ УНИВЕРСИТЕТ - ВАРНА,000083619,ЗАЛИВА 47-СП АД,103765686.0,company,55.0,Пряко / без обявление,2021-10-11,409033.504957,0,3,False,образование,Варна
3,e:00001-2021-0008:25147:_:eik:831079085:1,00001-2021-0008,„Снабдяване с природен газ за отопление на обе...,ИКОНОМИЧЕСКИ УНИВЕРСИТЕТ - ВАРНА,000083619,"""Примагаз"" АД",831079085.0,company,9.0,Пряко / без обявление,2021-12-15,76693.782179,0,1,True,образование,Варна
4,e:00001-2021-0009:29019:_:eik:130533432:1,00001-2021-0009,„Снабдяване с природен газ за отопление на обе...,ИКОНОМИЧЕСКИ УНИВЕРСИТЕТ - ВАРНА,000083619,"""ОВЕРГАЗ МРЕЖИ"" АД",130533432.0,company,9.0,Пряко / без обявление,2022-01-19,255645.940598,0,1,True,образование,Варна


## Section 2 - decisions & findings

**Cleaning decisions (all evidence-backed above):**

1. **Missing `bids_received` (9.9%) → dropped.** No target is possible without a
   bid count. The missingness is *concentrated by procedure* (range ~0%–13%, not
   the uniform 9.9% you'd see if it were random), so it is mildly MNAR. Logged as
   a limitation: results are conditional on a recorded bid count, and recording
   correlates with procedure type.
2. **Zero-bid rows (1.6%) → kept** in the negative class. Under
   `single_bid = (bids_received == 1)` they are genuinely "not single-bid."
   Keeping them shifts the positive rate only marginally (verified) and retains
   real low-competition signal. Dropping would have been a *different target*,
   not a cleaner version of this one.
3. **Authority join → `type_group` + `region` only.** Key is unique (no row
   multiplication) and coverage is complete. Aggregate columns
   (`spent_eur`, `contracts`, `suppliers`, `avg_eur`) excluded as leakage risks.
   `region` nulls (national-level bodies) filled with `"Unknown"` rather than
   dropped.

**Output:** `data/processed/contracts-modelling.csv` - the single source of
truth for every downstream section.

**Still open / deferred:**
- The **`procedure` near-circularity** ("Пряко / без обявление" is structurally
  single-source) is untouched here by design - it is investigated in section 3
  (model with and without it) and written up in Limitations.
- Encoding, the `value_eur` log transform, and date-feature engineering happen in
  sections 4–5, not here.

**Next (Section 3):** EDA on the cleaned table - single-bid rate by procedure,
sector, funding, authority type and time; and the formal procedure-circularity
investigation.